# Statics

In [1]:
Batch_size = 64

In [ ]:
import torch
import timm
from tqdm import tqdm
import random
from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image
import pandas as pd
import os
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score
from facenet_pytorch import InceptionResnetV1


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
print(torch.__version__)

# Dataset

## Dataset defenition

In [3]:

class CustomImageDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform_1=None,transform_2=None):
        """
        Args:
            csv_file (str): Path to the CSV file.
            root_dir (str): Directory with all the images.
            transform (callable, optional): Optional transform to be applied on a sample.
        """
        self.data = pd.read_csv(csv_file)
        self.root_dir = root_dir
        self.transform_1 = transform_1 if transform_1 else transforms.ToTensor()
        self.transform_2 = transform_2 if transform_2 else transforms.ToTensor()

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path = os.path.join(self.root_dir, self.data.iloc[idx]['path'])
        image = Image.open(img_path).convert("RGB")

        width, height = image.size

        # Manually define crop boxes
        split_x = int(width * 0.40)
        overlap = int(width*0.15)

        # Define crop boxes
        box1 = (0, 0, split_x+overlap, height)         # 25% left
        box2 = (split_x-overlap, 0, width, height)     # 75% right

        part1 = image.crop(box1)
        part2 = image.crop(box2)

        label = 1 if self.data.iloc[idx]['is_attack'] else 0  # attack=True -> 1, bonafide=False -> 0

        if self.transform_1 and self.transform_2:
            part1 = self.transform_1(part1)
            part2 = self.transform_2(part2)

        return part1 , part2 , label


## Load Dataset

In [4]:

transform_1 = transforms.Compose([
    transforms.Resize((160, 160)),
    transforms.ToTensor(),
    # transforms.Normalize([0.5]*3, [0.5]*3)
])

transform_2 = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    # transforms.Normalize([0.5]*3, [0.5]*3)
])

dataset_train = CustomImageDataset(csv_file='FantasyIDiap-ICCV25-Challenge/fantasyIDiap-train.csv',
                                   root_dir='FantasyIDiap-ICCV25-Challenge',
                                   transform_1=transform_1,
                                   transform_2=transform_2)
dataset_test = CustomImageDataset(csv_file='FantasyIDiap-ICCV25-Challenge/fantasyIDiap-test.csv',
                                  root_dir='FantasyIDiap-ICCV25-Challenge',
                                  transform_1=transform_1,
                                  transform_2=transform_2)

dataloader_train = DataLoader(dataset_train, batch_size=Batch_size, shuffle=True)
dataloader_test = DataLoader(dataset_test, batch_size=Batch_size, shuffle=False)


# Test Dataset

In [ ]:
for p1,p2, labels in dataloader_train:
    print(p1.shape)  # torch.Size([B, 3, 224, 224])
    print(torch.min(p1))  # torch.Size([B, 3, 224, 224])
    print(torch.max(p1))  # torch.Size([B, 3, 224, 224])
    print(p2.shape)  # torch.Size([B, 3, 224, 224])
    print(labels)        # tensor of shape [B]
    break

# print(len(dataset_train))
# print(len(dataset_test))

## Data Visualization

In [ ]:
def imshow(img, title=None):
    img = img.numpy().transpose((1, 2, 0))
    plt.imshow(img)
    if title:
        plt.title(title)
    plt.axis('off')

def show_random_samples(dataset, num_images=8):
    fig = plt.figure(figsize=(12, 6))
    indices = random.sample(range(len(dataset)), num_images)

    for i, idx in enumerate(indices):
        p1,p2, label = dataset[idx]
        label_name = 'Attack' if label == 1 else 'Bonafide'

        ax = fig.add_subplot(2, num_images // 2, i + 1)
        imshow(p1, title=f"{label_name}1")
    plt.tight_layout()
    plt.show()


show_random_samples(dataset_train, num_images=8)


# Model Defenition
<p>Implement your model here ...</p>

## FaceNet

In [7]:
class FaceNetForTextAnomaly(nn.Module):
    def __init__(self):
        super().__init__()
        self.facenet = InceptionResnetV1(pretrained='vggface2', classify=False)
        self.selector = nn.Linear(512,512)
        self.classifier = nn.Sequential(
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1),
            # nn.Sigmoid()  # For binary classification
        )

    def forward(self, x):
        x = self.facenet(x)
        features = self.selector(x)
        return self.classifier(features) , features

### Test FaceNet

In [8]:
# face_model = FaceNetForTextAnomaly()
# face_model = face_model.to(device)

# for p1,p2, labels in dataloader_train:
#     p1 = p1.to(device)
#     pred , out_1 = face_model(p1)
#     print(out_1.shape)
#     print(pred.shape)
#     print(pred[0])
#     break

## Vit

In [9]:
class ViTWithFeatures(nn.Module):
    def __init__(self):
        super().__init__()
        self.vit = timm.create_model('vit_base_patch16_224', pretrained=True)

        # Remove the default head
        self.feature_dim = 512
        self.feature_projector = nn.Linear(self.vit.head.in_features, self.feature_dim)

        # Classifier head (binary classification)
        self.classifier = nn.Sequential(
            nn.Linear(self.feature_dim, 1),
            # nn.Sigmoid()
        )

        # Remove the default head
        self.vit.reset_classifier(0)

    def forward(self, x):
        # Get CLS token output from ViT
        features = self.vit(x)  # shape: [B, vit_dim]
        feature_vector = self.feature_projector(features)  # [B, 512]
        out = self.classifier(feature_vector)              # [B, 1]
        return  out, feature_vector

### Test ViT

In [10]:
# text_model = ViTWithFeatures()
# text_model = text_model.to(device)

# for p1,p2, labels in dataloader_train:
#     p2 = p2.to(device)
#     pred , out_1 = text_model(p2)
#     print(out_1.shape)
#     print(pred.shape)
#     print(pred[0])
#     break

## Main Model

In [ ]:
class CombinedModelWithAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.facenet_branch = FaceNetForTextAnomaly()
        self.vit_branch = ViTWithFeatures()

        # Attention: use simple scaled dot-product attention
        self.attn = nn.MultiheadAttention(embed_dim=512, num_heads=4, batch_first=True)

        # Final classifier
        self.classifier = nn.Sequential(
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512,128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 1),
            # nn.Sigmoid()
        )

    def freeze_partial_backbones(self):
        # Freeze the first few blocks of FaceNet
        facenet_children = list(self.facenet_branch.facenet.children())
        for child in facenet_children[:6]:  # Adjust number based on actual layer names
            for param in child.parameters():
                param.requires_grad = False

        # Freeze the first few transformer blocks of ViT
        for name, module in self.vit_branch.vit.named_modules():
            if name.startswith("blocks.0") or name.startswith("blocks.1") or name.startswith("blocks.2"):  
                for param in module.parameters():
                    param.requires_grad = False

    def forward(self, p1,p2):
        _,f1 = self.facenet_branch(p1)  # [B, 512]
        _,f2 = self.vit_branch(p2)      # [B, 512]

        # Stack for attention: shape [B, 2, 512]
        stacked = torch.stack([f1, f2], dim=1)

        # Apply attention: Q=K=V=stacked → output shape: [B, 2, 512]
        attn_out, _ = self.attn(stacked, stacked, stacked)
        fused = attn_out.reshape(attn_out.size(0),1024)

        # Classify
        out = self.classifier(fused)
        return out, fused  # return both


In [12]:
model = CombinedModelWithAttention().to(device)
model.freeze_partial_backbones()

### Test MainModel

In [13]:
# for p1,p2, labels in dataloader_train:
#     p1 = p1.to(device)
#     p2 = p2.to(device)
#     pred , out = model(p1,p2)
#     print(pred.shape)
#     print(out.shape)
#     print(pred[0])
#     break

# Learning Functions

## f1 score calculator

In [14]:
def evaluate_f1_score(model, dataloader):
    model.eval()
    y_true = []
    y_pred = []

    with torch.no_grad():
        for p1,p2, labels in dataloader:
            p1,p2 = p1.to(device),p2.to(device)
            labels = labels.to(device).cpu().numpy()

            outputs,_ = model(p1,p2)
            # print(outputs)
            probs = torch.sigmoid(outputs).cpu().numpy()  # Convert logits to probabilities
            preds = (probs > 0.5).astype(int)

            y_true.extend(labels.astype(int))
            y_pred.extend(preds)

    f1 = f1_score(y_true, y_pred)
    return f1


## save model

In [15]:
def save_model(model,num_epochs,optimizer,optimizer_face,optimizer_vit,got_loss,name,score):
    model_state = {
        'num_epochs':num_epochs,
        'state_dict':model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'optimizer_face_state_dict': optimizer_face.state_dict(),
        'optimizer_vit_state_dict': optimizer_vit.state_dict(),
        'loss' : got_loss,
        'score' : score
    }

    torch.save(model_state,f'{name}-{num_epochs}.pth')
    print('_'*50)
    print(f'model saved at epoch {num_epochs} with name {name}-{num_epochs}.pth')
    print('_'*50)

## train an epoch

In [16]:
def train_one_epoch(model,train_loader,loss_fn,optimizer,epoch=None):
    epoch_loss = 0.0
    num_batches = 0
    model.train()
    with tqdm(train_loader,unit='batch') as tepoch:

        if epoch is not None:
            tepoch.set_description(f'Main => Epoch {epoch}')

        for images_p1,images_p2, labels in tepoch:
            images_p1,images_p2,labels = images_p1.to(device), images_p2.to(device),labels.to(device)

            # Zero the gradients.
            optimizer.zero_grad()

            # Feed forward
            out, _ = model(images_p1, images_p2)
            labels = labels.float().unsqueeze(1).to(device)


            # Calculate the batch loss.
            loss = loss_fn(out, labels)
            epoch_loss += loss.item()
            num_batches += 1

            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            tepoch.set_postfix(loss=loss.item())

    epoch_loss /= num_batches
    return model, epoch_loss


def train_one_epoch_face(model,train_loader,loss_fn,optimizer,epoch=None):
    epoch_loss = 0.0
    num_batches = 0
    model.train()
    with tqdm(train_loader,unit='batch') as tepoch:

        if epoch is not None:
            tepoch.set_description(f'Face => Epoch {epoch}')

        for images_p1,_, labels in tepoch:
            images_p1,labels = images_p1.to(device),labels.to(device)

            # Zero the gradients.
            optimizer.zero_grad()

            # Feed forward
            out, _ = model(images_p1)
            labels = labels.float().unsqueeze(1).to(device)


            # Calculate the batch loss.
            loss = loss_fn(out, labels)
            epoch_loss += loss.item()
            num_batches += 1

            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            tepoch.set_postfix(loss=loss.item())

    epoch_loss /= num_batches
    return model, epoch_loss


def train_one_epoch_vit(model,train_loader,loss_fn,optimizer,epoch=None):
    epoch_loss = 0.0
    num_batches = 0
    model.train()
    with tqdm(train_loader,unit='batch') as tepoch:

        if epoch is not None:
            tepoch.set_description(f'ViT => Epoch {epoch}')

        for _,images_p2, labels in tepoch:
            images_p2,labels = images_p2.to(device),labels.to(device)

            # Zero the gradients.
            optimizer.zero_grad()

            # Feed forward
            out, _ = model(images_p2)
            labels = labels.float().unsqueeze(1).to(device)


            # Calculate the batch loss.
            loss = loss_fn(out, labels)
            epoch_loss += loss.item()
            num_batches += 1

            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            tepoch.set_postfix(loss=loss.item())

    epoch_loss /= num_batches
    return model, epoch_loss

## validate an epoch

In [17]:
def validate_one_epoch(model,test_loader,loss_fn):
    model.eval()
    val_loss = 0.0
    val_batches = 0
    for idx, (images_p1,images_p2, labels) in enumerate(iter(test_loader)):
      images_p1,images_p2,labels = images_p1.to(device), images_p2.to(device),labels.to(device)
      labels = labels.float().unsqueeze(1).to(device)
      outputs,_ = model(images_p1,images_p2)
      loss = loss_fn(outputs,labels)
      val_loss += loss.item()
      val_batches += 1

    val_loss /= val_batches
    return val_loss

def validate_one_epoch_face(model,test_loader,loss_fn):
    model.eval()
    val_loss = 0.0
    val_batches = 0
    for idx, (images_p1,_, labels) in enumerate(iter(test_loader)):
      images_p1,labels = images_p1.to(device),labels.to(device)
      labels = labels.float().unsqueeze(1).to(device)
      outputs,_ = model(images_p1)
      loss = loss_fn(outputs,labels)
      val_loss += loss.item()
      val_batches += 1

    val_loss /= val_batches
    return val_loss


def validate_one_epoch_vit(model,test_loader,loss_fn):
    model.eval()
    val_loss = 0.0
    val_batches = 0
    for idx, (_,images_p2, labels) in enumerate(iter(test_loader)):
      images_p2,labels = images_p2.to(device),labels.to(device)
      labels = labels.float().unsqueeze(1).to(device)
      outputs,_ = model(images_p2)
      loss = loss_fn(outputs,labels)
      val_loss += loss.item()
      val_batches += 1

    val_loss /= val_batches
    return val_loss

# Lets Learn Somthing

In [18]:
criterion_main = nn.BCEWithLogitsLoss()
criterion_face = nn.BCEWithLogitsLoss()
criterion_vit = nn.BCEWithLogitsLoss()
optimizer_main = torch.optim.Adam(model.parameters(), lr=0.000001,weight_decay=1e-4)
optimizer_face = torch.optim.Adam(model.parameters(), lr=0.000001,weight_decay=1e-4)
optimizer_vit = torch.optim.Adam(model.parameters(), lr=0.000001,weight_decay=1e-4)

In [19]:
train_losses = []
test_losses = []
train_losses_face = []
test_losses_face = []
train_losses_vit = []
test_losses_vit = []
f1_scores = []
highest_score = 0
current_epoch = 0

In [ ]:
for _ in range(20):
    current_epoch+=1

    # Learning the Facenet
    model.facenet_branch , loss_train_face = train_one_epoch_face(model.facenet_branch,dataloader_train,criterion_face,optimizer_face,current_epoch) 
    train_losses_face.append(loss_train_face)
    loss_valid_face = validate_one_epoch_face(model.facenet_branch,dataloader_test,criterion_face)
    test_losses_face.append(loss_valid_face)

    
    # Learning the ViT
    model.vit_branch , loss_train_vit = train_one_epoch_vit(model.vit_branch,dataloader_train,criterion_vit,optimizer_vit,current_epoch) 
    train_losses_vit.append(loss_train_vit)
    loss_valid_vit = validate_one_epoch_vit(model.vit_branch,dataloader_test,criterion_vit)
    test_losses_vit.append(loss_valid_vit)


    # Learning whole model    
    model, loss_train = train_one_epoch(model,dataloader_train,criterion_main,optimizer_main,current_epoch)
    train_losses.append(loss_train)
    loss_valid = validate_one_epoch(model,dataloader_test,criterion_main)
    test_losses.append(loss_valid)

    f1 = evaluate_f1_score(model,dataloader_test)
    f1_scores.append(f1)

    print(f'FaceModel => Loss Train = {loss_train_face}, Loss Valid = {loss_valid_face}')
    print(f'ViT Model => Loss Train = {loss_train_vit}, Loss Valid = {loss_valid_vit}')
    print(f'Whole Model => Loss Train = {loss_train}, Loss Valid = {loss_valid} , f1 = {f1}')

    print('='*100)

    if current_epoch%20 == 0 :
        save_model(model,current_epoch,optimizer_main,optimizer_face,optimizer_vit,loss_valid,'all-new-fused-new-learn-final-decay',f1)
    if f1 > highest_score + 0.005:
        highest_score = f1
        save_model(model,current_epoch,optimizer_main,optimizer_face,optimizer_vit,loss_valid,'all-new-fused-goat-new-learn-final-decay',f1)

# Learning Curves and Statistics

## Learning curve

In [ ]:
plt.plot(train_losses, label = 'Train Loss')
plt.plot(test_losses, label = 'Validation Loss')
plt.title('Learning curves')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
plt.plot(train_losses_face, label = 'Train Loss')
plt.plot(test_losses_face, label = 'Validation Loss')
plt.title('Learning curves Face model')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
plt.plot(train_losses_vit, label = 'Train Loss')
plt.plot(test_losses_vit, label = 'Validation Loss')
plt.title('Learning curves ViT')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()

## Params and statistics

In [ ]:
def evaluate_f1_score(model, dataloader):
    model.eval()
    y_true = []
    y_pred = []

    with torch.no_grad():
        for p1,p2, labels in dataloader:
            p1,p2 = p1.to(device),p2.to(device)
            labels = labels.to(device).cpu().numpy()

            outputs,_ = model(p1,p2)
            probs = torch.sigmoid(outputs).cpu().numpy()  # Convert logits to probabilities
            preds = (probs > 0.5).astype(int)

            y_true.extend(labels.astype(int))
            y_pred.extend(preds)

    f1 = f1_score(y_true, y_pred)
    return f1


In [ ]:
f1 = evaluate_f1_score(model, dataloader_test)
print(f"F1 Score: {f1:.4f}")